<a href="https://colab.research.google.com/github/kwanda2426/projects/blob/main/social_media/Final_tweet_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Kwanda Mazibuko** - stdnr: 1077167

# **Importing Libraries**

In [5]:
!pip install -q tweepy gensim python-louvain

In [6]:
# Importing Libraries - make sure the packages are installed
import os
import tweepy as tw
import re
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from io import BytesIO

import networkx as nx
import community as community_louvain
from community.community_louvain import best_partition

import warnings
warnings.filterwarnings("ignore")

#making sure that we can see all rows and cols
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

# **Loading Data**

In [7]:
# Reading data
%%time
FILE_ID = "1lhoPhOLB4fm-IgXlKhZFfC0UiFfHSklZ"
xlsx_url = f"https://docs.google.com/spreadsheets/d/{FILE_ID}/export?format=xlsx"
r = requests.get(xlsx_url)
df_1 = pd.read_excel(BytesIO(r.content), engine = "openpyxl")


CPU times: user 1min 43s, sys: 1.27 s, total: 1min 44s
Wall time: 1min 52s


In [8]:
df_1.head(1)

,Query Id,Query Name,Date,Title,Url,Domain,Sentiment,Page Type,Language,Country Code,Continent Code,Continent,Country,City Code,Account Type,Added,Assignment,Author,Category Details,Checked,City,Display URLs,Entity Info,Expanded URLs,Facebook Author ID,Facebook Comments,Facebook Likes,Facebook Role,Facebook Shares,Facebook Subtype,Full Name,Full Text,Gender,Impressions,Instagram Comments,Instagram Followers,Instagram Following,Instagram Interactions Count,Instagram Posts,Interest,Last Assignment Date,Latitude,Location Name,Longitude,Media Filter,Media URLs,Mentioned Authors,Original Url,Priority,Professions,Resource Id,Short URLs,Starred,Station Name,Viewership,Status,Subtype,Thread Author,Thread Created Date,Thread Entry Type,Thread Id,Thread URL,Total Monthly Visitors,X Author ID,X Channel Role,X Followers,X Following,X Replies,X Reply to,X Repost of,X Reposts,X Likes,X Posts,X Verified,Updated,Reach (new),Publication Name,Licenses,Redacted,Redacted Fields,Redaction Reason,Asset Content Id,Asset Thumb Id,Author Verified Type,Avatar,Batch Id,Blog Name,Broadcast Media Url,Is Syndicated,Air Type,Broadcast Type,Media Type,Ad Value,Circulation,Region,Region Code,Daily Visitors,Engagement Type,Hashtags,Item Review,Kicker,Linkedin Comments,Linkedin Engagement,Linkedin Impressions,Linkedin Likes,Linkedin Shares,Linkedin Sponsored,Linkedin Video Views,Parent Post Id,Parent Blog Name,Pub Type,Publisher Sub Type,Rating,Reddit Score,Reddit Score Upvote Ratio,Reddit Comments,Reddit Author Karma,Root Post Id,Root Blog Name,Subreddit,Subreddit Subscribers,Subscriptions,Sub Title,React Score Overall,React Score Emotionality,React Score Harmful,Engagement Score,Subreddit NSFW,Reddit Post Flair,Reddit Author Flair,Subreddit Topics,Reddit Spoiler,Publication Id,Page Type Name,Content Source,Content Source Name,Custom,Bluesky Author Id,Bluesky Followers,Bluesky Following,Bluesky Likes,Bluesky Posts,Bluesky Quotes,Bluesky Replies,Bluesky Reposts,Can Edit Markup,Can Edit Metadata,Can Edit Segmentation,Can Edit Workflow,Copyright,Factiva Attribute Code,Has Full Text,Impact,Instagram Likes,Mention Id,Podcast Audience Estimate,Podcast Duration Ms,Raw Metadata,Reportable,Threads Likes,Threads Quotes,Threads Replies,Threads Reposts,Threads Shares,Threads Views,Tiktok Comments,Tiktok Connected Account,Tiktok Likes,Tiktok Reach,Tiktok Shares,Tiktok Views,Weblog Title,Youtube Comments,Youtube Duration Milliseconds,Youtube Favourites,Youtube Likes,Youtube Subscriber Count,Youtube Video Count,Emotion
0,2003594270,Kenya protests 2025,2025-08-31 21:59:50.0,RT @_James041 Aden Duale is as guilty as F.\n\nGuy has tried ethical card and it has failed.\n\nSHA theft has affected everyone and no one wants to be associated with a thief.\n\nHe has tried hiring affordable bloggers and they have all been humbled by the truth.\n\nThe more a crocodile smiles the more his anus widens.\n\nThere's no escape route for the wicked!\n\n#DualeMustGo #RutoMustGo #DrainTheSwamp,http://twitter.com/kelvinngari62/statuses/1962274155270635732,twitter.com,negative,twitter,en,KEN,AFRICA,Africa,Kenya,KEN.Coast.Mombasa,individual,2025-09-02T09:16:47.214+0000,NaN,kelvinngari62,NaN,False,Mombasa,NaN,"{entityId=13414952, entityConfidence=HIGH, url=https://www.wikidata.org/wiki/Q13414952}, {entityId=43169, entityConfidence=MEDIUM, url=https://www.wikidata.org/wiki/Q43169}, {entityId=2727213, entityConfidence=MEDIUM, url=https://www.wikidata.org/wiki/Q2727213}, {entityId=4682154, entityConfidence=LOW, url=https://www.wikidata.org/wiki/Q4682154}, {entityId=497, entityConfidence=LOW, url=https://www.wikidata.org/wiki/Q497}",NaN,NaN,0,0,NaN,0,NaN,kelvinngari62 (knn),RT @_James041 Aden Duale is as guilty as F.\n\nGuy has tried ethical card and it has failed.\n\nSHA theft has affected everyone and no one wants to be associated with a thief.\n\nHe has tried hiring affordable bloggers and they have all been humbled by the truth.\n\nThe more a crocodile smiles the more his anus widens.\n\nThere's no e

In [9]:
df_1.shape

(49823, 179)

#### **Pre-Processing**

In [10]:
# Changing column names
df_1.columns = df_1.columns.str.replace(' ', '_', regex=False).str.lower()

In [11]:
# Convert 'Date' column to datetime format
df_1['tweet_date'] = pd.to_datetime(df_1['date'], errors = 'coerce')

# June 2025 Tweets
df_temp = df_1[(df_1['tweet_date'].dt.year == 2025) & (df_1['tweet_date'].dt.month == 6)]

df_temp.reset_index(drop=True, inplace=True)

# final Data
df_final = df_temp[['author','date','title','full_text','tweet_date', 'sentiment','emotion','account_type','engagement_type','gender']]

In [12]:
df_final.head(1)

,author,date,title,full_text,tweet_date,sentiment,emotion,account_type,engagement_type,gender
0,Kapher_Oww,2025-06-30 23:30:45.0,"RT @MugureNjehia Amkeni....\n\nThey distracted us, passed the finance bill 2025.\nNow they are distracting us with their narrative of violence, anarchy, ethnic profiling so that we can forget that the police killed over 16 people on June 25th this year. \n\nVOTE OUT THIS REGIME ‼️ https://t.co/x33sqezV1F","RT @MugureNjehia Amkeni....\n\nThey distracted us, passed the finance bill 2025.\nNow they are distracting us with their narrative of violence, anarchy, ethnic profiling so that we can forget that the police killed over 16 people on June 25th this year. \n\nVOTE OUT THIS REGIME ‼️ https://t.co/x33sqezV1F",2025-06-30 23:30:45,negative,Sadness,individual,RETWEET,unknown


# **Question One**





In [13]:
# Extract user mentions
df_final['mentions'] = df_final['full_text'].apply(lambda x: re.findall(r'@(\w+)', str(x)))

# Extract retweets
df_final['retweet_user'] = df_final['full_text'].apply(lambda x: re.findall(r'^RT @(\w+)', str(x))[0] if re.match(r'^RT @(\w+)', str(x)) else None)

In [14]:
df_final.head()

,author,date,title,full_text,tweet_date,sentiment,emotion,account_type,engagement_type,gender,mentions,retweet_user
0,Kapher_Oww,2025-06-30 23:30:45.0,"RT @MugureNjehia Amkeni....\n\nThey distracted us, passed the finance bill 2025.\nNow they are distracting us with their narrative of violence, anarchy, ethnic profiling so that we can forget that the police killed over 16 people on June 25th this year. \n\nVOTE OUT THIS REGIME ‼️ https://t.co/x33sqezV1F","RT @MugureNjehia Amkeni....\n\nThey distracted us, passed the finance bill 2025.\nNow they are distracting us with their narrative of violence, anarchy, ethnic profiling so that we can forget that the police killed over 16 people on June 25th this year. \n\nVOTE OUT THIS REGIME ‼️ https://t.co/x33sqezV1F",2025-06-30 23:30:45,negative,Sadness,individual,RETWEET,unknown,[MugureNjehia],MugureNjehia
1,njoroge86,2025-06-30 23:29:31.0,RT @QuincyWandera Former Attorney General’s son was abducted and DCI and NIS denied that he was taken. It had to take a call to William for them to admit they have him and for William to order 🙄 his release. Unfortunately Ndiangui’s parents don’t have William’s phone number. #RutoMustGo https://t.co/wZlapGu1bc,RT @QuincyWandera Former Attorney General’s son was abducted and DCI and NIS denied that he was taken. It had to take a call to William for them to admit they have him and for William to order 🙄 his release. Unfortunately Ndiangui’s parents don’t have William’s phone number. #RutoMustGo https://t.co/wZlapGu1bc,2025-06-30 23:29:31,negative,Sadness,organisational,RETWEET,male,[QuincyWandera],QuincyWandera
2,FamacoFundi,2025-06-30 23:29:08.0,RT @Thuso1Africa Image of a youth in Kenya defying bullets to say enough of the corrupt government destroying their future. Africa must support the youth of Kenya. Williams Ruto and his corrupt government must go. #RutoMustGo @WilliamsRuto https://t.co/6kzTftppG4,RT @Thuso1Africa Image of a youth in Kenya defying bullets to say enough of the corrupt government destroying their future. Africa must support the youth of Kenya. Williams Ruto and his corrupt government must go. #RutoMustGo @WilliamsRuto https://t.co/6kzTftppG4,2025-06-30 23:29:08,negative,Sadness,individual,RETWEET,unknown,"[Thuso1Africa, WilliamsRuto]",Thuso1Africa
3,CazorlaRoba,2025-06-30 23:07:58.0,"RT @HonOscarSudi I've seen fiery debates on attacking police, but let's choose peace. Storming stations and seizing arms are unlawful. Gen Z protests were hijacked by gangs fueling chaos. https://t.co/fm8y8kwTqS","RT @HonOscarSudi I've seen fiery debates on attacking police, but let's choose peace. Storming stations and seizing arms are unlawful. Gen Z protests were hijacked by gangs fueling chaos. https://t.co/fm8y8kwTqS",2025-06-30 23:07:58,negative,Fear,individual,RETWEET,male,[HonOscarSudi],HonOscarSudi
4,Kapher_Oww,2025-06-30 23:02:39.0,RT @bevalynekwambo3 Happy New Month \n1. Arrest Lagat\n2. Impeach/Fire Murkomen \n3. Disband Ipoa\n4. Fire DCI Amin \n5. Justice for all extra judicial killings \n6. Ruto must go.,RT @bevalynekwambo3 Happy New Month \n1. Arrest Lagat\n2. Impeach/Fire Murkomen \n3. Disband Ipoa\n4. Fire DCI Amin \n5. Justice for all extra judicial killings \n6. Ruto must go.,2025-06-30 23:02:39,neutral,Joy,individual,RETWEET,unknown,[bevalynekwambo3],bevalynekwambo3


In [15]:
!pip -q install igraph lxml

In [16]:
import igraph as ig
df = df_final.copy()
mention_edges = []
for _, row in df.iterrows():
    author = row['author']
    for mentioned in row['mentions']:
        mention_edges.append((author, mentioned))

mention_graph = ig.Graph(directed=True)
mention_graph.add_vertices(list(set([u for u, v in mention_edges] + [v for u, v in mention_edges])))
mention_graph.add_edges(mention_edges)

In [17]:
retweet_edges = [(row['author'], row['retweet_user']) for _, row in df.iterrows() if row['retweet_user']]
retweet_graph = ig.Graph(directed=True)
retweet_graph.add_vertices(list(set([u for u, v in retweet_edges] + [v for u, v in retweet_edges])))
retweet_graph.add_edges(retweet_edges)

In [18]:
def compute_metrics(graph):
    degree = graph.degree()
    betweenness = graph.betweenness()
    clustering = graph.transitivity_local_undirected(vertices=None)

    return pd.DataFrame({
        'node': graph.vs['name'],
        'degree': degree,
        'betweenness': betweenness,
        'clustering': clustering
    }).sort_values(by='degree', ascending=False)

mention_metrics = compute_metrics(mention_graph)
retweet_metrics = compute_metrics(retweet_graph)

In [19]:
print("Top Mention Influencers:\n", mention_metrics.head(15))
print("Top Retweet Influencers:\n", retweet_metrics.head(15))

Top Mention Influencers:
                   node  degree   betweenness  clustering
3182         IAMRAPCHA     930      0.000000    0.000069
2623        FGaitho237     789  18913.881818    0.000966
11487      itskipronoh     720      0.000000    0.000104
7922       C_NyaKundiH     680  94893.179365    0.000316
7933      nickmuthumbi     551      0.000000    0.000240
3398      Honeyfarsafi     551  55636.751587    0.001023
5309           Ngartia     482  78667.422344    0.002397
804          McKennaMK     474      0.000000    0.000049
9042      WilliamsRuto     397      0.000000    0.001157
12866        _James041     349      0.000000    0.001147
4014       fit_ermined     337      0.000000    0.000590
6073       MissMutukuT     336      0.000000    0.000080
8886         Nyandia_G     330      0.000000    0.000079
11949  bevalynekwambo3     292      0.000000    0.000100
8759   sholard_mancity     290      0.000000    0.000052
Top Retweet Influencers:
                   node  degree   bet

In [21]:
mention_graph.write("mention_network.gexf", format='gexf')
retweet_graph.write("retweet_network.gexf", format='gexf')

OSError: unknown file format: gexf

Open these .gexf files in Gephi to visualize:
- Use ForceAtlas2 layout
- Color nodes by centrality
- Size nodes by degree
- Label top influencer


#### **Notes**:

- Each node is one of the top 10 most-mentioned accounts.
- Colour distinguishes unique users.
- Node size = how many times they were mentioned.
- Edges = mutual mentions among the top 10 (visibility interactions).
- Numeric labels = exact mention counts (shows relative dominance).

Mentions were chosen because, they represent active engagement, recognition by peers, amplification potential, and measurable authority — all of which are central to what it means to be influential in a social network.



# **Question Two**